# 🎤 RVC Voice Training trên Google Colab (đã sửa lỗi)
**Chạy từng cell theo thứ tự. GPU T4 miễn phí!**

> ⚠️ Vào **Runtime → Change runtime type → T4 GPU** trước khi chạy!

**Vì sao notebook này khác bản cũ?** Repo RVC chính thức đã được viết lại (nhánh Python 3.12, cấu trúc thư mục `assets/` mới, script `train/extract_f0_print.py` cũ đã đổi thành `train/dataset/extract_f0.py`, v.v.). Việc tự gọi từng script dòng lệnh như notebook cũ rất dễ vỡ vì phải tự dựng `config.json` và `filelist.txt` đúng logic nội bộ. Cách chính thức, ổn định nhất là cài đặt xong rồi **chạy `webui.py`** — giao diện Gradio này xử lý toàn bộ bước cắt dữ liệu / trích đặc trưng / train / lưu model. Notebook bên dưới chuẩn bị mọi thứ tự động, cell cuối mở giao diện đó.

In [ ]:
# ✅ CELL 1: Kiểm tra GPU
!nvidia-smi
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ Không có GPU!')

In [ ]:
# ✅ CELL 2: Mount Google Drive
# BUG CŨ: drive.mount('/content/drive') báo lỗi "Mountpoint must not already contain files"
# khi thư mục đã được mount từ trước (ví dụ chạy lại cell). Dùng force_remount=True để tránh lỗi này.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('✅ Drive đã mount!')

In [ ]:
# ✅ CELL 3: Cài đặt RVC (repo bản mới, nhánh Python 3.12)
import os, sys, shutil

%cd /content
if os.path.exists('/content/rvc'):
    shutil.rmtree('/content/rvc')

!git clone -q https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git rvc
%cd /content/rvc

# Thư viện hệ thống cần cho xử lý audio
!apt-get -qq install -y ffmpeg unzip libsndfile1 libportaudio2 > /dev/null

print('Python đang dùng:', sys.version)

# Repo yêu cầu Python 3.12 + Torch CUDA 11.8 (Colab T4 không phải RTX 50-series nên dùng cu118)
!python -m pip install -q torch==2.7.1+cu118 torchaudio==2.7.1+cu118 \
    --index-url https://download.pytorch.org/whl/cu118 \
    --extra-index-url https://pypi.org/simple

req_file = 'requirments_cu118_py312.txt'
if os.path.exists(req_file):
    !python -m pip install -q -r {req_file}
else:
    print(f'⚠️ Không tìm thấy {req_file}, cài từ requirements.txt mặc định thay thế.')
    !python -m pip install -q -r requirements.txt

!python -m pip install -q -U huggingface_hub

print('\n✅ Đã cài đặt xong RVC.')

In [ ]:
# ✅ CELL 4: Tải các model pretrained (đúng cấu trúc thư mục assets/ mới nhất của repo)
%cd /content/rvc

# hubert_base giờ là một thư mục (config.json + preprocessor_config.json + pytorch_model.bin),
# không còn là 1 file .pt duy nhất như bản cũ.
!hf download lj1995/VoiceConversionWebUI --revision main \
    --include "hubert_base/*" --local-dir assets

!hf download lj1995/VoiceConversionWebUI rmvpe.pt --revision main \
    --local-dir assets/rmvpe

!hf download lj1995/VoiceConversionWebUI --revision main \
    --include "pretrained/*" "pretrained_v2/*" --local-dir assets

# Các file "mute" (khoảng lặng) bắt buộc phải có để train, nằm trong mute.zip
!hf download lj1995/VoiceConversionWebUI mute.zip --revision main \
    --local-dir .model-downloads
!python -m zipfile -e .model-downloads/mute.zip logs

print('\n✅ Đã tải xong toàn bộ model pretrained!')

In [ ]:
# ✅ CELL 5: Chuẩn bị dataset từ Google Drive
# BUG CŨ: notebook gốc gán cứng đúng 1 tên file mp3, nếu bạn đổi tên file thì luôn báo "không tìm thấy".
# Bản sửa: tự quét toàn bộ Drive, copy MỌI file âm thanh tìm được vào thư mục dataset.
import os, shutil

MODEL_NAME = 'giong_toi'  # 👉 đổi tên model tại đây nếu muốn
dataset_dir = f'/content/rvc/dataset/{MODEL_NAME}'
os.makedirs(dataset_dir, exist_ok=True)

print('🔍 Đang quét Google Drive để tìm file âm thanh (.mp3 / .wav / .m4a / .flac)...')
found_files = []
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.lower().endswith(('.mp3', '.wav', '.m4a', '.flac')):
            found_files.append(os.path.join(root, file))

if not found_files:
    print('❌ Không tìm thấy file âm thanh nào trong Drive.')
    print('   Hãy tải file ghi âm giọng nói lên Google Drive rồi chạy lại cell này.')
else:
    for path in found_files:
        shutil.copy(path, os.path.join(dataset_dir, os.path.basename(path)))
    print(f'✅ Đã copy {len(found_files)} file vào {dataset_dir}:')
    for path in found_files:
        print('   🎵', path)
    print(f'\n🚀 Dataset đã sẵn sàng cho model "{MODEL_NAME}".')

## ✅ CELL 6: Mở giao diện huấn luyện (WebUI)

Chạy cell bên dưới rồi mở **link công khai (public URL)** mà nó in ra (dạng `https://xxxxx.gradio.live`). Trong giao diện:

1. Vào tab **Train** (huấn luyện) → nhập **Experiment name** = tên model bạn đặt ở CELL 5 (mặc định `giong_toi`).
2. Điền đường dẫn thư mục dataset: `/content/rvc/dataset/giong_toi` → bấm **Process data** (cắt dữ liệu).
3. Bấm **Feature extraction** (trích đặc trưng) — chọn phương pháp `rmvpe`, version `v2`.
4. Ở mục pretrained, để nguyên đường dẫn mặc định (`assets/pretrained_v2/f0G40k.pth`, `f0D40k.pth`).
5. Chỉnh `total_epoch`, `batch_size` theo ý muốn rồi bấm **Train model**.

Đây chính là cách chính thức mà repo khuyến nghị, tránh được toàn bộ lỗi "không tìm thấy script" mà notebook cũ gặp phải.

In [ ]:
# ✅ CELL 6: Chạy WebUI (--colab để tự tạo link public qua Gradio)
%cd /content/rvc
!python webui.py --colab

In [ ]:
# ✅ CELL 7: Lưu model đã train về Google Drive
# (Chạy cell này SAU KHI đã train xong ở giao diện WebUI của CELL 6)
import shutil, os

MODEL_NAME = 'giong_toi'  # phải trùng với tên đã dùng ở CELL 5 / lúc train
SAVE_TO_DRIVE = '/content/drive/MyDrive/RVC_Models'
os.makedirs(SAVE_TO_DRIVE, exist_ok=True)

model_dir = '/content/rvc/assets/weights'
logs_dir = f'/content/rvc/logs/{MODEL_NAME}'

saved_any = False
if os.path.exists(model_dir):
    for f in os.listdir(model_dir):
        if MODEL_NAME in f:
            shutil.copy(os.path.join(model_dir, f), os.path.join(SAVE_TO_DRIVE, f))
            print(f'✅ Đã lưu model: {f}')
            saved_any = True

if os.path.exists(logs_dir):
    for f in os.listdir(logs_dir):
        if f.endswith('.index'):
            shutil.copy(os.path.join(logs_dir, f), os.path.join(SAVE_TO_DRIVE, f))
            print(f'✅ Đã lưu index file: {f}')
            saved_any = True

if saved_any:
    print(f'\n🎉 Toàn bộ model đã được lưu vào: {SAVE_TO_DRIVE}')
else:
    print('⚠️ Không tìm thấy kết quả training để lưu. Hãy đảm bảo bạn đã train xong ở CELL 6.')